tables used:

processed - products, customers, geoloc, sellers

agg - orders_delivery_details, order_cost_detils, order_payments, order_review_summary

In [20]:
import pandas as pd
import duckdb
import os

conn = duckdb.connect('../database/olist.db')

# Order delivery

In [13]:
query = """
    SELECT * 
    FROM orders_delivery_details
"""

result = conn.execute(query).df().head()
print(result)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status  days_till_approval  days_till_shipped  days_till_delivery  \
0    delivered                   0                  2                   8   
1    delivered                   2                  2                  14   
2    delivered                   0                  0                   9   
3    delivered                   0                  4                  14   
4    delivered                   0                  1                   3   

   days_till_estim_delivery  delivery_accuracy  
0                        16                 -8  


In [14]:
# Let's focus on delivered orders for now
query = """
    CREATE OR REPLACE VIEW view_orders_delivered AS
SELECT
    *
FROM orders_delivery_details
WHERE order_status = 'delivered'

"""
conn.execute(query)


In [15]:
# summary statistics
query = """
    select 
        max(days_till_approval) as max_approval_days,
        avg(days_till_approval) as avg_approval_days,
        median(days_till_approval) as median_approval_days,
        max(days_till_shipped) as max_shipping_days,
        avg(days_till_shipped) as avg_shipping_days,
        median(days_till_shipped) as median_shipping_days,
        max(days_till_delivery) as max_days_till_delivery,
        avg(days_till_delivery) as avg_days_till_delivery,
        median(days_till_delivery) as median_days_till_delivery,
        max(days_till_estim_delivery) as max_days_estim_del,
        avg(days_till_estim_delivery) as avg_days_estim_del,
        median(days_till_estim_delivery) as median_days_estim_del,
        max(delivery_accuracy) as max_delivery_accuracy,
        avg(delivery_accuracy) as avg_delivery_accuracy,
        median(delivery_accuracy) as median_delivery_accuracy
    from view_orders_delivered
"""
result = conn.execute(query).df().head()
print(result)

   max_approval_days  avg_approval_days  median_approval_days  \
0                 31            0.51273                   0.0   

   max_shipping_days  avg_shipping_days  median_shipping_days  \
0                126           3.213918                   2.0   

   max_days_till_delivery  avg_days_till_delivery  median_days_till_delivery  \
0                     210               12.496849                       10.0   

   max_days_estim_del  avg_days_estim_del  median_days_estim_del  \
0                 156           24.372748                   24.0   

   max_delivery_accuracy  avg_delivery_accuracy  median_delivery_accuracy  
0                    188             -11.875889                     -12.0  


Conclusion:
-- data is right skewed

## Delayed order analysis

In [16]:
# Let's see which orders were delayed

query = """
    select 
        vod.order_id, customer_id, days_till_approval, days_till_shipped, days_till_delivery, 
        days_till_estim_delivery, delivery_accuracy, total_products, 
        total_cost, total_freight_value, total_cost_customer,
        shipping_burden
    from view_orders_delivered vod
    LEFT JOIN
    order_cost_detils ocd
    on vod.order_id = ocd.order_id
    where delivery_accuracy   > 0
    order by delivery_accuracy   desc

"""
result = conn.execute(query).df().head()
print(result)

                           order_id                       customer_id  \
0  1b3190b2dfa9d789e1f14c05b647a14a  d306426abe5fca15e54b645e4462dc7b   
1  ca07593549f1816d26a572e06dc1eab6  75683a92331068e2d281b11a7866ba44   
2  47b40429ed8cce3aee9199792275433f  cb2caaaead400c97350c37a3fc536867   
3  2fe324febf907e3ea3f2aa9650869fa5  65b14237885b3972ebec28c0f7dd2220   
4  285ab9426d6982034523a855f55a885e  9cf2c3fa2632cee748e1a59ca9d09b21   

   days_till_approval  days_till_shipped  days_till_delivery  \
0                   0                  3                 208   
1                   2                 15                 210   
2                   0                 34                 191   
3                   0                  4                 190   
4                   0                  1                 195   

   days_till_estim_delivery  delivery_accuracy  total_products  total_cost  \
0                        20                188               1      144.99   
1                   

In [115]:
# holistic view of orders delivered across cities and states

query = """
    select 
        odd.order_id, odd.days_till_delivery as delivery_days, c.customer_city as city, c.customer_state as state
    from view_orders_delivered odd
    LEFT JOIN customers c
    on odd.customer_id = c.customer_id
    

"""
# average delivery time across states
query = """ 
    SELECT 
        c.customer_state as state, c.customer_city as city,
        count(odd.order_id) as total_orders,
        round(AVG(odd.days_till_delivery),2) as avg_delivery_time,
        round(avg((delivery_accuracy)),2) as avg_delivery_accuracy,
        round(sum(ocd.total_cost_customer),2) as total_revenue_generated,
        round(avg(ocd.shipping_burden),2) as avg_shipping_burden
    FROM view_orders_delivered odd
    JOIN customers c ON odd.customer_id = c.customer_id
    LEFT JOIN order_cost_details as ocd
    on odd.order_id = ocd.order_id
    group by state, city
    order by total_orders desc
"""
result = conn.execute(query).df().head()
print("Average delivery time across states - \n ", result)

Average delivery time across states - 
    state            city  total_orders  avg_delivery_time  \
0    SP       sao paulo         15119               8.00   
1    RJ  rio de janeiro          6614              14.67   
2    MG  belo horizonte          2701              11.02   
3    DF        brasilia          2076              12.88   
4    PR        curitiba          1496              10.37   

   avg_delivery_accuracy  total_revenue_generated  avg_shipping_burden  
0                 -10.74               2107960.17                 0.30  
1                 -13.23               1111732.21                 0.37  
2                 -12.63                405950.51                 0.39  
3                 -12.06                345199.05                 0.39  
4                 -13.49                238459.72                 0.38  


In [113]:
# % of orders delivered late vs on time
query = """ 
    with on_time_orders as(
        SELECT 
            order_id, delivery_accuracy, 
        FROM view_orders_delivered
        where delivery_accuracy <= 0
    ),
    delayed_orders as (
        SELECT 
            order_id, delivery_accuracy
        FROM view_orders_delivered
        where delivery_accuracy > 0
    )
    select 
        count(ot.order_id) as total_ontime, 
        count(dor.order_id) as total_delayed,
        (total_ontime)/(total_ontime + total_delayed) as per_on_time,
        (total_delayed)/(total_ontime + total_delayed) as per_delayed,
    from on_time_orders as ot
    FULL OUTER JOIN delayed_orders as dor
    on ot.order_id = dor.order_id

"""
result = conn.execute(query).df().head()
print("% of orders delivered late vs on time - \n ", result)

% of orders delivered late vs on time - 
     total_ontime  total_delayed  per_on_time  per_delayed
0         89936           6534     0.932269     0.067731


Therefore, the shipping limit breach is 0.06% that is 6,534 orders were delivered after the expected delivery time.

In [197]:
# Let's look at the orders that got delayed
query = """ 
    CREATE OR REPLACE VIEW view_delayed_orders_holistic AS
        with orders_holistic_data as(
            select 
                vod.order_id, vod.customer_id, 
                oi.product_id, product_category_name as category, product_weight_g as weight,
                price, freight_value, shipping_burden,
                days_till_delivery as delivered_in_days, delivery_accuracy as delayed_by_days, 
                customer_city as city, customer_state as state
            from view_orders_delivered vod
            LEFT JOIN order_items oi
            on vod.order_id = oi.order_id
            LEFT JOIN products p
            on oi.product_id = p.product_id
            LEFT JOIN customers c
            on vod.customer_id = c.customer_id
            LEFT JOIN order_cost_details ocd
            on vod.order_id = ocd.order_id
        ),
        delayed_orders as (
            select * from orders_holistic_data
            where delayed_by_days > 0
        )
        select * from delayed_orders
        
"""


In [203]:
# Is weight causing the delay in shipment? How many orders weight heavier?
query = """
    select
        avg(weight) as average_weight,
        median(weight) as median_weight,
        min(weight) as min_weight,
        max(weight) as max_weight
    from delayed_orders_holistic

"""
result = conn.execute(query).df().head()
print(" \n ", result)

 
     average_weight  median_weight  min_weight  max_weight
0     2460.717924          800.0          50       40425


The data is clearly skewed. The data has outliers in terms of weight. Making judgement based on average would be the incorrect choice for a rightly skweed data. Hence, choosing the median.

In [18]:
conn.close()